# Cardiomegaly Classifier — Training Notebook

**Backbone:** `torchxrayvision` DenseNet-121 (`densenet121-res224-all`) — pretrained on ~1M chest X-rays. We fine-tune the Cardiomegaly output on our data with a multi-seed ensemble, AMP, 6-pass TTA, and bootstrap-stabilised threshold tuning on val.

**Grading:** composite = 0.5·AUC + 0.25·sensitivity + 0.25·specificity. Threshold is tuned on validation to maximise sens + spec (Youden's J).

This notebook is intentionally minimal — all logic lives in `src/`.

| File | Responsibility |
|------|----------------|
| `src/config.py` | `Config` dataclass + global `CFG` |
| `src/utils.py` | `set_seed`, `free_device_cache`, `log_run` |
| `src/data.py` | `load_labels`, `split_dataframe` |
| `src/dataset.py` | `ChestXrayDataset`, `TTADataset`, `SubmissionDataset`, `xrv_normalize_np` |
| `src/transforms.py` | `make_transforms`, `make_tta_transforms` |
| `src/model.py` | `build_model` (xrv DenseNet), `cardio_logit`, freeze helpers |
| `src/train.py` | `train_ensemble`, `tta_predict_ensemble`, threshold helpers, `save_results` |

## 1 · Environment Setup (Colab + Git)

**Google Colab users:** just run all cells top to bottom — git clone, dependencies and OneDrive sync are all handled automatically.  
**Local users:** the Colab-specific blocks are skipped automatically.

In [ ]:
import os
import sys
import subprocess

# ── Detect Google Colab ──────────────────────────────────────────────────────
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # ── 1. Clone or pull the repository ─────────────────────────────────────
    REPO_URL    = "https://github.com/jcvilar19/heart-scan-helper.git" 
    REPO_FOLDER = "/content/heart-scan-helper"

    if not os.path.exists(REPO_FOLDER):
        print("Cloning repository...")
        subprocess.run(["git", "clone", REPO_URL, REPO_FOLDER], check=True)
    else:
        print("Pulling latest changes...")
        subprocess.run(["git", "-C", REPO_FOLDER, "pull", "origin", "main"], check=True)

    # ── 2. Change working directory to notebooks/ ────────────────────────────
    os.chdir(f"{REPO_FOLDER}/model_training/notebooks")
    print(f"Working directory: {os.getcwd()}")

    # ── 3. Install Python dependencies ───────────────────────────────────────
    print("Installing dependencies...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"],
        check=True,
    )
    print("✓ Dependencies installed")

# ── Add src/ to Python path (works for both Colab and local) ────────────────
sys.path.insert(0, os.path.abspath(".."))

from src.config import CFG
from src.utils import set_seed

set_seed(CFG.seed)
print(f"\nDevice  : {CFG.device}")
print(f"img_size: {CFG.img_size}")
print(f"Results : {CFG.output_dir}")

### 1.1 · Google Drive Setup

Results are saved to a **shared Google Drive folder** after training so every team member can compare all runs in one place.

**One-time setup per person:**
1. Ask the team lead to share the Google Drive folder with your Google account (edit access)
2. Go to [drive.google.com](https://drive.google.com) → "Shared with me" → right-click the folder → **"Add shortcut to My Drive"**
3. Note the path you placed it at (e.g. `cardiomegaly-results`)
4. Update `GDRIVE_RESULTS_FOLDER` in the cell below

**Every session:** Google Drive is mounted automatically when you run this cell.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    print("Google Drive mounted ✓")

    # ── Path to the shared Google Drive folder ───────────────────────────────
    # Update this to match where you placed the shortcut in your Google Drive.
    GDRIVE_RESULTS_FOLDER = "/content/drive/MyDrive/cardiomegaly-results"  # ← UPDATE THIS

    os.makedirs(GDRIVE_RESULTS_FOLDER, exist_ok=True)
    print(f"Shared results folder: {GDRIVE_RESULTS_FOLDER}")
else:
    GDRIVE_RESULTS_FOLDER = None
    print("Running locally — Google Drive sync will be skipped")

## 2 · Data Loading & Splitting

In [ ]:
from src.data import load_labels, split_dataframe

full_df                   = load_labels(CFG.csv_path, CFG.image_dir)
train_df, val_df, test_df = split_dataframe(full_df)

print(f"Train: {len(train_df)}  pos={int(train_df['label'].sum())}")
print(f"Val  : {len(val_df)}  pos={int(val_df['label'].sum())}")
print(f"Test : {len(test_df)}  pos={int(test_df['label'].sum())}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  CONFIG — change settings here and re-run this cell before training
#  No kernel restart needed. CFG is a shared singleton — all modules see changes.
# ═══════════════════════════════════════════════════════════════════════════════
from src.config import CFG

# ── Backbone ─────────────────────────────────────────────────────────────────
# "densenet121"        ← torchxrayvision DenseNet-121, pretrained on ~1M chest X-rays ✓
# "rad-dino"           ← microsoft/rad-dino, DINOv2 ViT-B/14 pretrained on ~1M chest X-rays ✓
#                         img_size is auto-set to 518 (native ViT-B/14 resolution)
#                         frozen_blocks: 0–12 (12 transformer blocks total)
# "mobilenet_v3_large" ← ImageNet pretrained, faster
# "efficientnet_b0"    ← ImageNet pretrained, balanced
# "efficientnet_b3"    ← ImageNet pretrained, higher accuracy
CFG.backbone = "rad-dino"

# ── Ensemble vs. single model ────────────────────────────────────────────────
CFG.use_ensemble = True          # False = single model with CFG.seed (faster)
CFG.seed         = 7          # used when use_ensemble=False
CFG.seeds        = [8, 7, 2024, 849]  # used when use_ensemble=True

# ── Training schedule ────────────────────────────────────────────────────────
CFG.frozen_epochs       = 10
CFG.finetune_epochs     = 15
CFG.early_stop_patience = 4
# frozen_blocks: how many backbone blocks to keep frozen in Stage 2
#   DenseNet-121  → 0–4   dense block groups   (0 = unfreeze all)
#   RAD-DINO ViT  → 0–12  transformer blocks   (8 = keep lower 8 frozen, last 4 trainable)
CFG.frozen_blocks = 0 if CFG.backbone == "rad-dino" else 0

# ── Optimiser ────────────────────────────────────────────────────────────────
# Head LR: MLP head in both stages.
#   DenseNet-121 / torchvision backbones : 3e-4 – 1e-3
#   RAD-DINO ViT                         : 1e-4 – 3e-4  (ViT heads converge with lower LR)
CFG.head_lr = 1e-4 if CFG.backbone == "rad-dino" else 5e-4
# Backbone LR: Stage 2 only.  For ViT keep 50–100× below head_lr!
#   DenseNet-121 / torchvision backbones : 1e-4 – 3e-4
#   RAD-DINO ViT                         : 1e-5  (higher values destroy pretrained features)
CFG.backbone_lr = 1e-4 if CFG.backbone == "rad-dino" else 2e-4

# ── Mixup augmentation ────────────────────────────────────────────────────────
# Interpolates two training samples and their labels:  x_mix = λ·x_i + (1-λ)·x_j
# λ ~ Beta(α, α).  0 = disabled.  Typical: 0.2 – 0.4.
# Helps with: overfitting on small datasets, sharpening decision boundaries.
CFG.mixup_alpha = 0

# ── Label smoothing ───────────────────────────────────────────────────────────
# Softens hard targets: y_smooth = y*(1-ε) + 0.5*ε.  0 = disabled.  Typical: 0.05 – 0.10.
# Helps with: overconfident predictions, poor calibration.
CFG.label_smoothing = 0

# ── Loss function ─────────────────────────────────────────────────────────────
# False → standard BCEWithLogitsLoss (default, stable)
# True  → α·BCE + (1-α)·(1 - soft_composite)  where soft_composite ≈ 0.5·AUC + 0.25·sens + 0.25·spec
#         Directly optimises the evaluation metric; may improve threshold-sensitive scores.
CFG.use_composite_loss   = False
CFG.composite_loss_alpha = 0.5    # blend: 0 = pure composite, 1 = pure BCE
CFG.composite_loss_gamma = 1.0    # pairwise-sigmoid temperature (higher → sharper AUC gradient)

# ── img_size: auto-set from backbone, override here if needed ────────────────
# RAD-DINO native resolution: 518 × 518 (ViT-B/14 patch size = 14 px → 37×37 patches)
# Use any multiple of 14 if memory is tight: 392, 308, 224.
_auto_img_size = {
    "densenet121":          224,
    "densenet121-res224-all": 224,
    "rad-dino":             518,   # ViT-B/14 native — use multiple of 14
    "mobilenet_v3_large":   224,
    "efficientnet_b0":      224,
    "efficientnet_b3":      300,
}
CFG.img_size = _auto_img_size.get(CFG.backbone, 224)
# CFG.img_size = 392   # override: smaller multiple-of-14 size for rad-dino if memory is tight

os.makedirs(CFG.output_dir, exist_ok=True)
print(f"  backbone    : {CFG.backbone}")
print(f"  img_size    : {CFG.img_size}")
print(f"  use_ensemble: {CFG.use_ensemble}  seeds: {CFG.seeds}")
print(f"  frozen_blocks: {CFG.frozen_blocks}  |  stage1: {CFG.frozen_epochs}ep  stage2: {CFG.finetune_epochs}ep")
print(f"  device      : {CFG.device}")
_loss_str = (
    f"SoftComposite  (α={CFG.composite_loss_alpha}, γ={CFG.composite_loss_gamma})"
    if CFG.use_composite_loss else "BCEWithLogitsLoss"
)
print(f"  loss        : {_loss_str}")

## 3 · Datasets & DataLoaders

In [ ]:
from torch.utils.data import DataLoader
from src.dataset import ChestXrayDataset, get_normalize_fn
from src.transforms import make_transforms

# Always rebuilt from current CFG — re-run this cell whenever backbone or img_size changes
train_tf, eval_tf = make_transforms(CFG.img_size)

train_ds = ChestXrayDataset(train_df, pil_transform=train_tf, use_erasing=True, backbone=CFG.backbone)
val_ds   = ChestXrayDataset(val_df,   pil_transform=eval_tf,                    backbone=CFG.backbone)
test_ds  = ChestXrayDataset(test_df,  pil_transform=eval_tf,                    backbone=CFG.backbone)

_kw = dict(
    batch_size=CFG.batch_size,
    num_workers=CFG.num_workers,
    pin_memory=(CFG.device == "cuda"),
    persistent_workers=CFG.num_workers > 0,
)
train_loader = DataLoader(train_ds, shuffle=True,  **_kw)
val_loader   = DataLoader(val_ds,   shuffle=False, **_kw)
test_loader  = DataLoader(test_ds,  shuffle=False, **_kw)

_norm_fn = get_normalize_fn(CFG.backbone).__name__
print(f"Backbone     : {CFG.backbone}  →  normalisation: {_norm_fn}")
print(f"img_size     : {CFG.img_size}")
print(f"Batches      — train: {len(train_loader)}  val: {len(val_loader)}  test: {len(test_loader)}")

## 4 · Sanity check augmentation (optional)

Visualise a few training augmentation samples to confirm the pipeline works end-to-end.

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax in axes[0]:
    pil = Image.open(train_df.sample(1, random_state=random.randint(0, 9999)).iloc[0]["image_path"]).convert("L")
    ax.imshow(pil, cmap="gray"); ax.set_title("original"); ax.axis("off")
for ax in axes[1]:
    row = train_df.sample(1, random_state=random.randint(0, 9999)).iloc[0]
    img = Image.open(row["image_path"]).convert("L")
    img = train_tf(img)
    ax.imshow(img, cmap="gray"); ax.set_title(f"aug  label={row['label']}"); ax.axis("off")
plt.tight_layout(); plt.show()

## 5 · Training

Set `CFG.use_ensemble = True` to train one model per seed in `CFG.seeds` (predictions are averaged in logit space).  
Set `CFG.use_ensemble = False` to train a single model with `CFG.seed` (faster experimentation).

Change the backbone with `CFG.backbone`: `"densenet121"` | `"mobilenet_v3_large"` | `"efficientnet_b0"` | `"efficientnet_b3"`

In [ ]:
from src.train import train

models, history = train(
    train_loader, val_loader,
    output_dir=CFG.output_dir,
)

## 6 · TTA Evaluation & Bootstrap Threshold Selection

Composite score (matches the grading rubric):

\[ \text{score} = 0.5 \cdot \text{AUC} + 0.25 \cdot \text{sensitivity} + 0.25 \cdot \text{specificity} \]

Threshold is selected on val to maximise `sens + spec`, stabilised with a bootstrap so it generalises better than a single-shot pick.

In [ ]:
from sklearn.metrics import roc_auc_score
from src.train import (
    tta_predict, tta_predict_ensemble,
    metrics_at_threshold, find_best_threshold,
    bootstrap_threshold, select_threshold,
)

label = f"Ensemble ({len(models)} seeds)" if CFG.use_ensemble else f"Single model (seed={CFG.seed})"



# ── Combined val predictions (averaged in logit space) ───────────────────
print(f"\n=== {label} — val ===")
val_out     = tta_predict_ensemble(models, val_df)
val_auc_ens = roc_auc_score(val_out["y_true"], val_out["y_prob"])
print(f"Val AUC: {val_auc_ens:.4f}")

# ── Threshold tuning on val (composite-driven) ───────────────────────────
best_threshold, m_single, m_boot = select_threshold(val_out["y_true"], val_out["y_prob"])
print(f"Val single-shot threshold: {m_single['threshold']:.4f}  composite={m_single['composite']:.4f}")
print(f"Val bootstrap threshold  : {m_boot['threshold']:.4f}  composite={m_boot['composite']:.4f}")
print(f"Chosen threshold         : {best_threshold:.4f}")

# ── Test predictions ─────────────────────────────────────────────────────
print(f"\n=== {label} — test ===")
test_out = tta_predict_ensemble(models, test_df)

val_metrics  = metrics_at_threshold(val_out["y_true"],  val_out["y_prob"],  best_threshold)
test_metrics = metrics_at_threshold(test_out["y_true"], test_out["y_prob"], best_threshold)

print("\n---------- VAL ----------")
for k, v in val_metrics.items():
    print(f"  {k:12s}: {v}")
print("\n---------- TEST ---------")
for k, v in test_metrics.items():
    print(f"  {k:12s}: {v}")

print(f"\n*** TEST COMPOSITE SCORE: {test_metrics['composite']:.4f} ***")
print(f"    (AUC={test_metrics['auc']:.4f}, sens={test_metrics['sensitivity']:.4f}, spec={test_metrics['specificity']:.4f})")

## 7 · ROC Curves

In [ ]:
import os
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, out, title in [(axes[0], val_out, "Validation"), (axes[1], test_out, "Test")]:
    fpr, tpr, _ = roc_curve(out["y_true"], out["y_prob"])
    ax.plot(fpr, tpr, label=f"AUC = {auc(fpr, tpr):.4f}")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"{title} ROC Curve")
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CFG.output_dir, "roc_curves.png"), dpi=150)
plt.show()

## 8 · Save Results

In [ ]:
from src.train import save_results

# Auto-generated from CFG — override manually if needed
_mode = f"ensemble{len(models)}seed" if CFG.use_ensemble else "single"
MODEL_NAME = f"{CFG.backbone}_{_mode}"

save_results(
    models_list=models,
    history=history,
    val_out=val_out,
    test_out=test_out,
    best_threshold=best_threshold,
    output_dir=CFG.output_dir,
    model_name=MODEL_NAME,
    config=CFG,
)

## 9 · Sync Results to Shared Google Drive

Results are copied to the shared Google Drive folder automatically.  
Every team member's run is merged into the shared `results_log.csv` so you can compare all runs in one place.

In [ ]:
import shutil
import pandas as pd

if IN_COLAB and GDRIVE_RESULTS_FOLDER:
    gdrive_log_path = os.path.join(GDRIVE_RESULTS_FOLDER, "results_log.csv")

    # ── Merge results_log.csv with the shared one ────────────────────────────
    local_log  = pd.read_csv("results_log.csv") if os.path.exists("results_log.csv") else pd.DataFrame()
    remote_log = pd.read_csv(gdrive_log_path)   if os.path.exists(gdrive_log_path)   else pd.DataFrame()

    merged_log = (
        pd.concat([remote_log, local_log], ignore_index=True)
          .drop_duplicates(subset=["run_id"])
    )
    merged_log.to_csv("results_log.csv", index=False)
    merged_log.to_csv(gdrive_log_path, index=False)
    print(f"results_log.csv merged  ({len(merged_log)} total runs)")

    # ── Copy run results folder to Google Drive ──────────────────────────────
    run_dest = os.path.join(GDRIVE_RESULTS_FOLDER, MODEL_NAME)
    shutil.copytree(CFG.output_dir, run_dest, dirs_exist_ok=True)

    print(f"✓ Run artefacts  → {run_dest}")
    print(f"✓ results_log.csv → {gdrive_log_path}")

elif IN_COLAB:
    print("⚠️  Google Drive folder not set. Check GDRIVE_RESULTS_FOLDER in Section 1.1.")
else:
    print(f"Running locally — results are in: {CFG.output_dir}")

## 10 · Submission (optional)

Only needed if unlabelled test images exist at `CFG.submission_test_dir`.

In [ ]:
import os, pandas as pd
from src.train import predict_submission

if os.path.isdir(CFG.submission_test_dir):
    submission_out = predict_submission(models, CFG.submission_test_dir)
    sub_df = pd.DataFrame({
        "image_file": submission_out["names"],
        "prob":       submission_out["y_prob"],
    })
    sub_df["pred"] = (sub_df["prob"] >= best_threshold).astype(int)
    sub_path = os.path.join(CFG.output_dir, "daily_submission.csv")
    sub_df.to_csv(sub_path, index=False)
    print(f"Submission saved → {sub_path}  ({len(sub_df)} rows)")
    print(f"Positive rate    : {sub_df['pred'].mean():.3f}")
    display(sub_df.head())
else:
    print(f"No submission directory found at: {CFG.submission_test_dir}")